# Mushroom Dataset - Baseline Evaluation

## Contributors
- Liam: baseline evaluation and validation design

This notebook establishes a majority-class baseline and records the PyCaret automated comparison. The holdout test set is not used for model selection.

## Train, validation, and test design

The data is split into 80% training data and 20% stratified holdout test data. The holdout remains untouched until final evaluation. PyCaret uses cross-validation inside the training data, so model selection does not use the final test set.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
df = pd.read_csv("mushroom_cleaned.csv")
target = "class"
X = df.drop(columns=target)
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train shape: {X_train.shape}; test shape: {X_test.shape}")

## Majority-class baseline

This is the minimum benchmark that a useful classifier should beat.

In [ ]:
majority_class = y_train.mode().iloc[0]
majority_predictions = np.full(len(y_test), majority_class)
baseline_results = pd.DataFrame([{
    "model": "Majority class",
    "accuracy": accuracy_score(y_test, majority_predictions),
    "balanced_accuracy": balanced_accuracy_score(y_test, majority_predictions),
    "f1_poisonous": f1_score(y_test, majority_predictions, pos_label="p", zero_division=0),
}])
baseline_results

## PyCaret automated comparison

PyCaret compares candidate classifiers using cross-validation on the training split. Balanced Accuracy is registered explicitly because it is the selection metric for this imbalanced, safety-sensitive problem.

In [ ]:
from pycaret.classification import add_metric, compare_models, pull, setup
from sklearn.metrics import balanced_accuracy_score

pycaret_train = X_train.copy()
pycaret_train[target] = y_train
setup(
    data=pycaret_train,
    target=target,
    session_id=RANDOM_STATE,
    fold=5,
    numeric_imputation="median",
    categorical_imputation="most_frequent",
    verbose=False,
    html=False,
)
add_metric(
    id="balanced_accuracy",
    name="Balanced Accuracy",
    score_func=balanced_accuracy_score,
    target="pred",
    greater_is_better=True,
)
best_model = compare_models(sort="Balanced Accuracy", n_select=1)
pycaret_comparison_table = pull()
pycaret_comparison_table.head(10)

## Baseline conclusion

The majority baseline is only a reference point. The selected PyCaret model must be compared against it on the untouched holdout set, with poisonous recall and balanced accuracy receiving priority over accuracy alone.